# Train Commutative CNN

This notebook intentionally keeps orchestration thin. Split preparation, fitting, reporting, plotting, and persistence are delegated to shared utilities in `src.ml`.


In [1]:
%load_ext autoreload
%autoreload 2

from dataclasses import asdict
from pathlib import Path

import pandas as pd

from src.ml import (
    LossWeightConfig,
    OptimizationConfig,
    display_experiment_summary,
    display_holdout_evaluation,
    fit_estimator_on_experiment,
    persist_experiment_artifacts,
    plot_holdout_branch_embedding_projections,
    plot_training_history,
    prepare_multitask_experiment_data,
)
from src.dataset_config import load_current_dataset_artifact_path
from src.tensor_utils import build_tensor_embedding_2d, load_labeled_tensor_dataset, plot_tensor_embedding_2d

pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)
pd.set_option("display.width", None)
pd.set_option("display.expand_frame_repr", False)


In [2]:
from src.ml import CommutativeCNNClassifier, CommutativeCNNConfig


In [3]:
# User inputs

dataset_artifact_path = load_current_dataset_artifact_path()
experiment_output_dir = Path("artifacts/nb7_commutative_cnn")
persist_artifacts = True

holdout_fraction = 0.25
validation_fraction_within_train = 0.20
train_num_random_rotations = 6
rotation_range_degrees = 12.0

model_config = CommutativeCNNConfig(
    spatial_conv_channels=(8, 12, 16),
    spatial_kernel_size_z=(1, 1, 1),
    spatial_kernel_size_xy=(5, 3, 3),
    spatial_stride_z=(1, 1, 1),
    spatial_stride_xy=(1, 1, 1),
    spatial_pool_kernel_z=(1, 1, 1),
    spatial_pool_kernel_xy=(2, 2, 2),
    spatial_pool_stride_z=(1, 1, 1),
    spatial_pool_stride_xy=(2, 2, 2),
    temporal_st_channels=(24, 24),
    temporal_st_kernel_sizes=(3, 3),
    temporal_ts_channels=(16, 16),
    temporal_ts_kernel_sizes=(5, 5),
    spatial_agg_channels=(16, 32),
    spatial_agg_kernel_size_z=(3, 3),
    spatial_agg_kernel_size_xy=(3, 3),
    spatial_agg_stride_z=(1, 1),
    spatial_agg_stride_xy=(1, 1),
    spatial_agg_pool_kernel_z=(1, 1),
    spatial_agg_pool_kernel_xy=(2, 2),
    spatial_agg_pool_stride_z=(1, 1),
    spatial_agg_pool_stride_xy=(2, 2),
    patch_size_z=1,
    patch_size_xy=8,
    embedding_dim=16,
    num_prototypes=12,
    dropout=0.45,
)
optimization_config = OptimizationConfig(
    batch_size=8,
    epochs=20,
    learning_rate=2e-4,
    weight_decay=3e-3,
    early_stopping_patience=4,
    early_stopping_min_delta=0.0,
    scheduler_patience=1,
    scheduler_factor=0.5,
    scheduler_min_lr=1e-6,
    validation_split=0.0,
    random_state=0,
    standardize=True,
    device=None,
    verbose=True,
)
loss_weight_config = LossWeightConfig(
    action_weight=1.0,
    compound_weight=0.2,
    concentration_weight=0.2,
    consistency_weight=0.15,
    feature_weight=0.01,
    prototype_temperature=0.1,
)


In [4]:
dataset = load_labeled_tensor_dataset(dataset_artifact_path)


In [5]:
experiment = prepare_multitask_experiment_data(
    dataset,
    holdout_fraction=holdout_fraction,
    validation_fraction_within_train=validation_fraction_within_train,
    train_num_random_rotations=train_num_random_rotations,
    rotation_range_degrees=rotation_range_degrees,
    random_state=optimization_config.random_state,
)


In [6]:
display_experiment_summary(experiment)


,split,n_samples
0,train_augmented,245
1,train_base,35
2,val,9
3,holdout,15


,mechanism_of_action,compound,concentration_band,n_samples
0,NMDAR_Activation,(RS)-(Tetrazol-5-yl)glycine,control,42
1,GABAAR_NegativeAllostericModulator,DMCM,control,35
2,NMDAR_Antagonist,Ketamine,high,35
3,GABAAR_Antagonist,Bemegride,high,28
4,GABAAR_NegativeAllostericModulator,DMCM,high,28
5,NMDAR_Activation,(RS)-(Tetrazol-5-yl)glycine,high,28
6,NMDAR_Antagonist,Ketamine,control,28
7,GABAAR_Antagonist,Bemegride,control,21


In [7]:
model = CommutativeCNNClassifier(
    model_config=model_config,
    optimization_config=optimization_config,
    loss_weight_config=loss_weight_config,
)


In [ ]:
fit_estimator_on_experiment(model, experiment)


cols:
    ep=epoch
    trL=train_loss trA=train_action_loss
    trCC=train_commutative_consistency_loss
    trFA=train_feature_alignment_loss
    trCo=train_compound_loss trCn=train_concentration_loss
    vaL=val_loss vaA=val_action_loss
    vaCC=val_commutative_consistency_loss
    vaFA=val_feature_alignment_loss
    vaCo=val_compound_loss vaCn=val_concentration_loss
    lr=learning_rate eta=estimated_time_remaining
     ep      trL      trA     trCC     trFA     trCo     trCn      vaL      vaA     vaCC     vaFA     vaCo     vaCn       lr       eta
001/020   3.2502   1.6002   7.5726   1.2131   1.7702   0.7399   2.6168   1.6326   3.3610   0.1317   1.6551   0.7386 2.00e-04     19:59
002/020   2.8655   1.5889   5.1995   0.9927   1.7009   0.7333   2.4328   1.5419   2.7299   0.2323   1.6575   0.7380 2.00e-04     18:19
003/020   2.6966   1.5276   4.5731   0.8390   1.6555   0.7179   2.3164   1.4888   2.3503   0.2664   1.6253   0.7368 2.00e-04     16:58
004/020   2.5469   1.4868   3.9051   0.

In [ ]:
plot_training_history(model, title="Commutative CNN loss curves", loess_frac=0.6);


In [ ]:
holdout_evaluation = display_holdout_evaluation(model, experiment)


In [ ]:
holdout_embedding_projection = build_tensor_embedding_2d(
    model.transform(experiment.splits.X_holdout),
    experiment.y_true_holdout["action"],
    label_map=experiment.label_maps["action"],
    metadata=experiment.splits.metadata_holdout,
    method="pca",
    random_state=optimization_config.random_state,
)
plot_tensor_embedding_2d(
    holdout_embedding_projection,
    title="Holdout embedding projection by action",
    marker_column="compound",
)


In [ ]:
holdout_branch_embedding_projections = plot_holdout_branch_embedding_projections(model, experiment)


In [ ]:
run_config = {
    "dataset_artifact_path": dataset_artifact_path,
    "holdout_fraction": holdout_fraction,
    "validation_fraction_within_train": validation_fraction_within_train,
    "train_num_random_rotations": train_num_random_rotations,
    "rotation_range_degrees": rotation_range_degrees,
    "model_config": asdict(model_config),
    "optimization_config": asdict(optimization_config),
    "loss_weight_config": asdict(loss_weight_config),
}


In [ ]:
if persist_artifacts:
    experiment_artifacts = persist_experiment_artifacts(
        output_dir=experiment_output_dir,
        estimator=model,
        reports=holdout_evaluation.reports,
        config=run_config,
    )
